In [1]:
import numpy as np
import h5py
import tensorflow as tf
import random
import math

In [2]:
train_data_file="train_chunk.h5"
test_data_file="test_chunk.h5"

In [3]:
hf_train = h5py.File(train_data_file, 'r')
print("hf_train", hf_train)
print("keys", hf_train.keys())

hf_test = h5py.File(test_data_file, 'r')
print("hf_test", hf_test)
print("keys", hf_test.keys())

"""Load in the data and labels from the HDF5 files and convert them to TensorFlow tensors""" 

data_train = tf.convert_to_tensor(np.array(hf_train['features']), dtype=tf.float32)
labels_train = tf.convert_to_tensor(np.array(hf_train['source_file']), dtype=tf.string)

data_test = tf.convert_to_tensor(np.array(hf_test['features']), dtype=tf.float32)
labels_test = tf.convert_to_tensor(np.array(hf_test['source_file']), dtype=tf.string)

feature_names = tf.convert_to_tensor(np.array(hf_train['feature_names']), dtype=tf.string)
print("feature_names", feature_names)
print("data_train shape:", data_train.shape)
print("labels_train shape:", labels_train.shape)
print("data_test shape:", data_test.shape)
print("labels_test shape:", labels_test.shape)




hf_train <HDF5 file "train_chunk.h5" (mode r)>
keys <KeysViewHDF5 ['feature_names', 'features', 'source_file']>
hf_test <HDF5 file "test_chunk.h5" (mode r)>
keys <KeysViewHDF5 ['feature_names', 'features', 'source_file']>
feature_names tf.Tensor(
[b'mLL' b'ptLL' b'dPhi_LL' b'dPhiLLmet' b'MET' b'mt' b'goodjet_n'
 b'goodbjet_n' b'Lepton1_Pt' b'Lepton1_Eta' b'Lepton1_E' b'Lepton1_Phi'
 b'Lepton1_charge' b'Lepton1_type' b'Lepton2_Pt' b'Lepton2_Eta'
 b'Lepton2_E' b'Lepton2_Phi' b'Lepton2_charge' b'Lepton2_type' b'weight'], shape=(21,), dtype=string)
data_train shape: (200000, 21)
labels_train shape: (200000,)
data_test shape: (20000, 21)
labels_test shape: (20000,)


In [4]:
import h5py
import numpy as np

# =========================================================
# Input files
# =========================================================


X_train = np.array(hf_train["features"], dtype=np.float32)
X_test = np.array(hf_test["features"], dtype=np.float32)

feature_names = [
    x.decode("utf-8") if isinstance(x, bytes) else str(x)
    for x in np.array(hf_train["feature_names"])
]

print("feature_names:", feature_names)

col_idx = {name: i for i, name in enumerate(feature_names)}

# =========================================================
# Output file names
# =========================================================
train_token_file = "train_binned_tokens.h5"
test_token_file = "test_binned_tokens.h5"

# =========================================================
# Bin edges
# =========================================================
BIN_EDGES = {
    "mLL": np.array([10, 17, 22, 27, 31, 34, 38, 42, 45, 50, 54, 55], dtype=np.float32),
    "ptLL": np.array([30, 43, 47, 51, 55, 59, 64, 69, 77, 90, 145, 312], dtype=np.float32),
    "dPhi_LL": np.linspace(0.0, np.pi, 11, dtype=np.float32),
    "dPhiLLmet": np.linspace(0.0, np.pi, 11, dtype=np.float32),
    "MET": np.array([20, 30, 37, 43, 48, 52, 58, 64, 72, 85, 130, 876], dtype=np.float32),
    "mt": np.array([40, 75, 85, 93, 99, 106, 113, 121, 132, 148, 205, 545], dtype=np.float32),
    "goodjet_n": np.array([-0.5, 0.5, 1.5], dtype=np.float32),
    "Lepton1_Pt": np.array([25, 29, 32, 35, 37, 40, 44, 48, 54, 65, 112, 291], dtype=np.float32),
    "Lepton1_Eta": np.linspace(-2.5, 2.5, 11, dtype=np.float32),
    "Lepton1_E": np.array([25, 36, 43, 50, 58, 68, 80, 99, 126, 170, 338, 833], dtype=np.float32),
    "Lepton1_Phi": np.linspace(-np.pi, np.pi, 13, dtype=np.float32),
    "Lepton1_charge": np.array([-1.5, 0.0, 1.5], dtype=np.float32),
    "Lepton1_type": np.array([10.0, 12.0, 14.0], dtype=np.float32),
    "Lepton2_Pt": np.array([15, 17, 18.5, 20, 22, 24, 26, 28, 31, 35, 51, 103], dtype=np.float32),
    "Lepton2_Eta": np.linspace(-2.5, 2.5, 11, dtype=np.float32),
    "Lepton2_E": np.array([15, 21, 25, 29, 33, 39, 46, 56, 72, 97, 178, 390], dtype=np.float32),
    "Lepton2_Phi": np.linspace(-np.pi, np.pi, 13, dtype=np.float32),
    "Lepton2_charge": np.array([-1.5, 0.0, 1.5], dtype=np.float32),
    "Lepton2_type": np.array([10.0, 12.0, 14.0], dtype=np.float32),
}

# Order of features that will be tokenized and saved
TOKEN_FEATURES = [
    "mLL",
    "ptLL",
    "dPhi_LL",
    "dPhiLLmet",
    "MET",
    "mt",
    "goodjet_n",
    "Lepton1_Pt",
    "Lepton1_Eta",
    "Lepton1_E",
    "Lepton1_Phi",
    "Lepton1_charge",
    "Lepton1_type",
    "Lepton2_Pt",
    "Lepton2_Eta",
    "Lepton2_E",
    "Lepton2_Phi",
    "Lepton2_charge",
    "Lepton2_type",
]

# =========================================================
# Build token offsets
# =========================================================
offsets = {}
vocab_size = 0

for feat in TOKEN_FEATURES:
    offsets[feat] = vocab_size
    vocab_size += len(BIN_EDGES[feat]) - 1

print("vocab_size:", vocab_size)
print("offsets:", offsets)

# =========================================================
# Tokenization
# =========================================================
def tokenize_features(X_np, feature_names, token_features, bin_edges_dict, offsets_dict):
    col_idx_local = {name: i for i, name in enumerate(feature_names)}
    token_cols = []

    for feat in token_features:
        values = X_np[:, col_idx_local[feat]]
        edges = bin_edges_dict[feat]

        bin_idx = np.digitize(values, edges, right=False) - 1
        bin_idx = np.clip(bin_idx, 0, len(edges) - 2)

        token_ids = offsets_dict[feat] + bin_idx
        token_cols.append(token_ids.astype(np.int32))

    return np.stack(token_cols, axis=1)

train_tokens = tokenize_features(X_train, feature_names, TOKEN_FEATURES, BIN_EDGES, offsets)
test_tokens = tokenize_features(X_test, feature_names, TOKEN_FEATURES, BIN_EDGES, offsets)

print("train_tokens shape:", train_tokens.shape)
print("test_tokens shape:", test_tokens.shape)
print("example row:", train_tokens[0])

# Keep weights separately if present
train_weights = X_train[:, col_idx["weight"]].astype(np.float32) if "weight" in col_idx else None
test_weights = X_test[:, col_idx["weight"]].astype(np.float32) if "weight" in col_idx else None

# =========================================================
# Extract source_file if present
# =========================================================
train_source = None
test_source = None

if "source_file" in hf_train:
    train_source = np.array(hf_train["source_file"])

if "source_file" in hf_test:
    test_source = np.array(hf_test["source_file"])

# =========================================================
# Save new HDF5 files
# =========================================================
with h5py.File(train_token_file, "w") as f:
    f.create_dataset("features", data=train_tokens, dtype="i4")
    f.create_dataset("feature_names", data=np.array(TOKEN_FEATURES, dtype="S"))
    f.create_dataset("vocab_size", data=np.array([vocab_size], dtype=np.int32))

    if train_weights is not None:
        f.create_dataset("weight", data=train_weights, dtype="f4")

    if train_source is not None:
        f.create_dataset("source_file", data=train_source)

with h5py.File(test_token_file, "w") as f:
    f.create_dataset("features", data=test_tokens, dtype="i4")
    f.create_dataset("feature_names", data=np.array(TOKEN_FEATURES, dtype="S"))
    f.create_dataset("vocab_size", data=np.array([vocab_size], dtype=np.int32))

    if test_weights is not None:
        f.create_dataset("weight", data=test_weights, dtype="f4")

    if test_source is not None:
        f.create_dataset("source_file", data=test_source)

print(f"Saved tokenized train file: {train_token_file}")
print(f"Saved tokenized test file:  {test_token_file}")
# Optional cleanup
hf_train.close()
hf_test.close()

feature_names: ['mLL', 'ptLL', 'dPhi_LL', 'dPhiLLmet', 'MET', 'mt', 'goodjet_n', 'goodbjet_n', 'Lepton1_Pt', 'Lepton1_Eta', 'Lepton1_E', 'Lepton1_Phi', 'Lepton1_charge', 'Lepton1_type', 'Lepton2_Pt', 'Lepton2_Eta', 'Lepton2_E', 'Lepton2_Phi', 'Lepton2_charge', 'Lepton2_type', 'weight']
vocab_size: 162
offsets: {'mLL': 0, 'ptLL': 11, 'dPhi_LL': 22, 'dPhiLLmet': 32, 'MET': 42, 'mt': 53, 'goodjet_n': 64, 'Lepton1_Pt': 66, 'Lepton1_Eta': 77, 'Lepton1_E': 87, 'Lepton1_Phi': 98, 'Lepton1_charge': 110, 'Lepton1_type': 112, 'Lepton2_Pt': 114, 'Lepton2_Eta': 125, 'Lepton2_E': 135, 'Lepton2_Phi': 146, 'Lepton2_charge': 158, 'Lepton2_type': 160}
train_tokens shape: (200000, 19)
test_tokens shape: (20000, 19)
example row: [  3  15  24  39  48  58  64  68  80  88 107 111 113 119 130 137 157 158
 160]
Saved tokenized train file: train_binned_tokens.h5
Saved tokenized test file:  test_binned_tokens.h5
